# **My Music Revisits:** An Exploratory Data Analysis of My Music Streaming Behavior

## Purpose
In this notebook, approximately 6-months of my personal music streaming data is used to understand and explore my listening behavior, discovery habits, repetition patterns, time-based trends, source routes, session dynamics, track lifecycles, and recommendation opportunities. The analysis is built from processed Parquet tables generated by the project pipeline. The goal is to understand my behavior as a listener, assess the quality of the data present, and use the insights found here to develop an interactive dashboard with a built in recommendation system.

## Imports and Configuration

In [ ]:
import warnings
from music_analytics.config import LISTENER_NAME
from pathlib import Path

import numpy
import pandas
import plotly.express as plot
import plotly.graph_objects as graph
import plotly.io as plot_io
from IPython.display import HTML, Markdown, display

warnings.filterwarnings("ignore")

In [ ]:
pandas.set_option("display.max_columns", 160)
pandas.set_option("display.max_rows", 120)
pandas.set_option("display.width", 220)
pandas.set_option("display.float_format", lambda value: f"{value:,.3f}")

In [ ]:
blue_set = ["#dff4ff", "#bde8ff", "#8fd3ff", "#5ab8ff", "#2f93e6", "#1769aa", "#0d3f66"]

blue_theme = graph.layout.Template(
    layout=graph.Layout(
        colorway=blue_set,
        font=dict(family="Inter, Segoe UI, Arial", color="#18324a"),
        paper_bgcolor="rgba(238, 248, 255, 0.70)",
        plot_bgcolor="rgba(248, 252, 255, 0.72)",
    )
)

plot_io.templates["blue_glass"] = blue_theme
plot_io.templates.default = "blue_glass"

page_style = """
<style>
body, .jp-Notebook, .notebook_app {
    background: linear-gradient(135deg, #eaf7ff 0%, #f7fbff 45%, #dcefff 100%) !important;
}
.glass-title {
    padding: 28px 30px;
    margin: 10px 0 22px 0;
    border: 1px solid rgba(86, 158, 215, 0.28);
    border-radius: 22px;
    background: linear-gradient(135deg, rgba(255, 255, 255, 0.72), rgba(209, 235, 255, 0.46));
    box-shadow: 0 18px 45px rgba(43, 116, 171, 0.16);
    backdrop-filter: blur(14px);
}
.glass-title h1 {
    margin: 0;
    color: #0e426d;
    font-size: 34px;
    letter-spacing: -0.04em;
}
.glass-title p {
    margin: 8px 0 0 0;
    color: #38627f;
    font-size: 16px;
}
</style>
"""

display(HTML(page_style))
display(
    HTML(
        "<div class='glass-title'>"
        f"<h1>{LISTENER_NAME}’s Music Revisits</h1>"
        "<h4>An Exploratory Data Analysis of My Music Streaming Behavior</h4>"
        "</div>"
    )
)

In [ ]:
# depends on device for now, might remove later
def find_root():
    here = Path.cwd()

    for path in [here, *here.parents]:
        if (path / "data").exists():
            return path

    mac = Path.home() / "Documents" / "Coding" / "music-analytics"

    if mac.exists():
        return mac

    if Path("/mnt/data").exists():
        return Path("/mnt/data")

    return here

In [ ]:
def read_table(name):
    paths = [
        proc_root / name,
        proc_root / "analytics" / name,
        data_root / "processed" / name,
        chat_root / name,
        root / name,
    ]

    for path in paths:
        if path.exists():
            data = pandas.read_parquet(path)
            print(f"Loaded {name} with {len(data):,} rows and {data.shape[1]:,} columns.")
            return data

    print(f"Missing {name}.")
    return pandas.DataFrame()

In [ ]:
root = find_root()
data_root = root / "data"
proc_root = data_root / "processed"
chat_root = Path("/mnt/data")
report_root = root / "reports"
eda_root = report_root / "exploratory_data_analysis"
fig_root = report_root / "figures" / "exploratory_data_analysis"
tab_root = report_root / "tables" / "exploratory_data_analysis"

for folder in [eda_root, fig_root, tab_root]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {root}.")
print(f"EDA Figure Directory: {fig_root}.")
print(f"EDA Table Directory: {tab_root}.")

In [ ]:
albums = read_table("albums.parquet")
artists = read_table("artists.parquet")
daily_tracks = read_table("daily_tracks.parquet")
days = read_table("days.parquet")
features = read_table("features.parquet")
genres = read_table("genres.parquet")
hours = read_table("hours.parquet")
library = read_table("library.parquet")
life = read_table("life.parquet")
playlists = read_table("playlists.parquet")
playlist_tracks = read_table("playlist_tracks.parquet")
plays = read_table("plays.parquet")
profile = read_table("profile.parquet")
candidates = read_table("recommend_candidates.parquet")
recommend_profile = read_table("recommend_profile.parquet")
recommendations = read_table("recommendations.parquet")
sessions = read_table("sessions.parquet")
sources = read_table("sources.parquet")
from music_analytics.recommend import join_pair_details
recommendations = join_pair_details(recommendations, candidates)


## Helper Functions

In [ ]:
def plain_time(series):
    time = pandas.to_datetime(series, errors="coerce")

    if getattr(time.dt, "tz", None) is not None:
        time = time.dt.tz_convert(None)

    return time

In [ ]:
def set_time(data, cols):
    data = data.copy()

    for col in cols:
        if col in data.columns:
            data[col] = plain_time(data[col])

    return data

In [ ]:
def has_cols(data, cols):
    return all(col in data.columns for col in cols)

In [ ]:
def rate_text(value):
    if pandas.isna(value):
        return "0.0%"

    return f"{value:.1%}"

In [ ]:
def number_text(value):
    if pandas.isna(value):
        return "0"

    if abs(float(value)) >= 1000:
        return f"{value:,.0f}"

    return f"{value:,.2f}"

In [ ]:
def safe_mean(data, col):
    if data.empty or col not in data.columns:
        return numpy.nan

    return data[col].mean()

In [ ]:
def top_name(data, name, by):
    if data.empty or name not in data.columns or by not in data.columns:
        return "Unknown"

    row = data.sort_values(by, ascending=False).head(1)

    if row.empty:
        return "Unknown"

    return str(row[name].iloc[0])

In [ ]:
def show_table(data, title, rows=12):
    display(Markdown(f"### {title}"))

    if data.empty:
        print(f"{title} is empty.")
        return

    view = data.head(rows)

    view = view.style.set_table_styles(
        [
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "separate"),
                    ("border-spacing", "0"),
                    ("border", "1px solid rgba(86, 158, 215, 0.28)"),
                    ("border-radius", "14px"),
                    ("overflow", "hidden"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("background", "rgba(206, 234, 255, 0.72)"),
                    ("color", "#0e426d"),
                    ("font-weight", "700"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("background", "rgba(255, 255, 255, 0.58)"),
                    ("color", "#18324a"),
                ],
            },
        ]
    )

    display(view)

In [ ]:
def style_chart(chart, title, subtitle="", height=520):
    full_title = f"<b>{title}</b>"

    if subtitle:
        full_title = f"<b>{title}</b><br><sup>{subtitle}</sup>"

    chart.update_layout(
        title=dict(text=full_title, x=0.02, xanchor="left"),
        height=height,
        margin=dict(l=55, r=35, t=88, b=55),
        hoverlabel=dict(
            bgcolor="rgba(241, 249, 255, 0.96)",
            bordercolor="#7dbde8",
            font_size=12,
            font_family="Inter, Segoe UI, Arial",
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
        ),
    )

    chart.update_xaxes(
        showgrid=True,
        gridcolor="rgba(52, 119, 172, 0.16)",
        zeroline=False,
        title_font=dict(color="#244f73"),
    )

    chart.update_yaxes(
        showgrid=True,
        gridcolor="rgba(52, 119, 172, 0.16)",
        zeroline=False,
        title_font=dict(color="#244f73"),
    )

    return chart

In [ ]:
def save_chart(chart, name):
    path = fig_root / f"{name}.html"
    chart.write_html(path)
    print(f"Saved chart to {path}.")

    return path

In [ ]:
def show_chart(chart, name, title, subtitle="", height=520):
    chart = style_chart(chart, title, subtitle, height)
    chart.show()
    save_chart(chart, name)

    return chart

## Load Datasets

In [ ]:
plays = set_time(plays, ["played_at", "start_at", "received_at", "date"])
days = set_time(days, ["date"])
daily_tracks = set_time(daily_tracks, ["date", "played_at"])
features = set_time(features, ["first_play", "last_play", "added_at", "release_at"])
life = set_time(life, ["first_play", "last_play"])
library = set_time(library, ["added_at", "release_at"])
sessions = set_time(sessions, ["start", "end"])
artists = set_time(artists, ["first_play", "last_play"])
albums = set_time(albums, ["first_play", "last_play"])

In [ ]:
tables = {
    "albums": albums,
    "artists": artists,
    "daily_tracks": daily_tracks,
    "days": days,
    "features": features,
    "genres": genres,
    "hours": hours,
    "library": library,
    "life": life,
    "playlists": playlists,
    "playlist_tracks": playlist_tracks,
    "plays": plays,
    "profile": profile,
    "recommend_candidates": candidates,
    "recommend_profile": recommend_profile,
    "recommendations": recommendations,
    "sessions": sessions,
    "sources": sources,
}

## Data Scope and Quality
Before creating the plots and tables, it is important to understand the scale, scope, and quality of the data present. Understanding the scope of the data helps provide context for every insight that follows. This section measures the size of the tables, checks for missing values, and assesses the completeness of the datasets.

In [ ]:
scope_rows = []

for name, data in {
    "plays": plays,
    "features": features,
    "artists": artists,
    "albums": albums,
    "genres": genres,
    "days": days,
    "hours": hours,
    "sessions": sessions,
    "sources": sources,
    "life": life,
    "library": library,
    "playlists": playlists,
    "playlist_tracks": playlist_tracks,
    "recommend_candidates": candidates,
    "recommendations": recommendations,
}.items():

    rows = len(data)
    cols = data.shape[1]

    missing = int(data.isna().sum().sum())

    total_cells = rows * cols

    missing_share = (
        missing / total_cells
        if total_cells > 0
        else 0
    )

    duplicate_rows = int(data.duplicated().sum())

    scope_rows.append(
        {
            "table": name,
            "rows": rows,
            "columns": cols,
            "missing_cells": missing,
            "missing_share": missing_share,
            "duplicate_rows": duplicate_rows,
        }
    )

scope = pandas.DataFrame(scope_rows)

scope = scope.sort_values(
    "rows",
    ascending=False,
)

### Data Scope Summary

In [ ]:
display(scope)

In [ ]:
print(f"Total Tables Loaded: {len(scope):,}.")
print(f"Total Rows Across All Tables: {scope['rows'].sum():,}.")
print(f"Total Columns Across All Tables: {scope['columns'].sum():,}.")

### Listening History Range

In [ ]:
if "played_at" in plays.columns:

    first_play = plays["played_at"].min()
    last_play = plays["played_at"].max()

    listening_days = (
        last_play - first_play
    ).days

    listening_years = (
        listening_days / 365.25
    )

    print(f"First Play: {first_play}.")
    print(f"Last Play: {last_play}.")
    print(f"Listening Window: {listening_days:,} days.")
    print(f"Listening Window: {listening_years:.2f} years.")

### Personal Listening Scale

In [ ]:
print(f"Listening Events: {len(plays):,}.")

if "track_key" in plays.columns:
    print(f"Unique tracks: {plays['track_key'].nunique():,}.")

if "artist" in plays.columns:
    print(f"Unique artists: {plays['artist'].nunique():,}.")

if "album" in plays.columns:
    print(f"Unique Albums: {plays['album'].nunique():,}.")

if "genre" in plays.columns:
    print(f"Unique Genres: {plays['genre'].nunique():,}.")

if "date" in plays.columns:
    print(f"Active Listening Days: {plays['date'].nunique():,}.")

### Recommendation System Scale

In [ ]:
print(f"Feature table rows: {len(features):,}.")
print(f"Recommendation candidates: {len(candidates):,}.")
print(f"Saved recommendation pairs: {len(recommendations):,}.")

## Data Overview

In [ ]:
total_tracks = (
    plays["track_key"].nunique()
    if "track_key" in plays.columns
    else 0
)

total_artists = (
    plays["artist"].nunique()
    if "artist" in plays.columns
    else 0
)

total_days = (
    plays["date"].nunique()
    if "date" in plays.columns
    else 0
)

In [ ]:
print(
    f"This dataset contains {len(plays):,} listening events "
    f"across {total_tracks:,} tracks, "
    f"{total_artists:,} artists, "
    f"and {total_days:,} listening days."
)

In [ ]:
total = len(plays)

start = plays["played_at"].min() if "played_at" in plays.columns and not plays.empty else pandas.NaT
end = plays["played_at"].max() if "played_at" in plays.columns and not plays.empty else pandas.NaT

minutes = plays["minutes"].sum() if "minutes" in plays.columns and not plays.empty else 0
listening_hours = minutes / 60

track_count = plays["track_key"].nunique() if "track_key" in plays.columns else 0
artist_count = plays["artist"].nunique() if "artist" in plays.columns else 0
genre_count = plays["genre"].nunique() if "genre" in plays.columns else 0
album_count = plays["album"].nunique() if "album" in plays.columns else 0
active_days = plays["date"].nunique() if "date" in plays.columns else len(days)

full_rate = safe_mean(plays, "full_play")
friction_rate = safe_mean(plays, "friction")
repeat_rate = safe_mean(plays, "repeat_seen")
first_rate = safe_mean(plays, "first_play")
retained_rate = safe_mean(features, "retained")

In [ ]:
top_artist = top_name(artists, "artist", "plays")
top_genre = top_name(genres, "genre", "plays")
top_source = top_name(sources, "source_label", "plays")
top_session = top_name(sessions, "session_type", "plays") if "plays" in sessions.columns else "Unknown"
top_block = plays["time_block"].value_counts().idxmax() if "time_block" in plays.columns and not plays.empty else "Unknown"

In [ ]:
if repeat_rate >= 0.60:
    listener = "a return-driven listener"
elif first_rate >= 0.45:
    listener = "an exploration-driven listener"
else:
    listener = "a balanced listener"

In [ ]:
if friction_rate >= 0.60:
    selectivity = "highly selective"
elif friction_rate >= 0.35:
    selectivity = "moderately selective"
else:
    selectivity = "low-friction"

In [ ]:
if full_rate >= 0.40:
    finish_style = "often let songs finish"
elif full_rate >= 0.20:
    finish_style = "sometimes let songs finish"
else:
    finish_style = "rarely let songs finish"

In [ ]:
identity = pandas.DataFrame(
    [
        {"metric": "Listening events", "value": f"{total:,}"},
        {"metric": "Listening window", "value": f"{start:%Y-%m-%d} to {end:%Y-%m-%d}" if pandas.notna(start) and pandas.notna(end) else "Unknown"},
        {"metric": "Active days", "value": f"{active_days:,}"},
        {"metric": "Listening hours", "value": f"{listening_hours:,.1f}"},
        {"metric": "Tracks heard", "value": f"{track_count:,}"},
        {"metric": "Artists heard", "value": f"{artist_count:,}"},
        {"metric": "Albums heard", "value": f"{album_count:,}"},
        {"metric": "Genres heard", "value": f"{genre_count:,}"},
        {"metric": "Full listen rate", "value": rate_text(full_rate)},
        {"metric": "Friction rate", "value": rate_text(friction_rate)},
        {"metric": "Repeat seen rate", "value": rate_text(repeat_rate)},
        {"metric": "First play rate", "value": rate_text(first_rate)},
        {"metric": "Retained track rate", "value": rate_text(retained_rate)},
    ]
)

identity.to_csv(tab_root / "exploratory_listening_identity.csv", index=False)

In [ ]:
print(f"I am a {listener}, and your behavior is {selectivity}.")
print(f"I {finish_style}, with a full listen rate of {rate_text(full_rate)} and a friction rate of {rate_text(friction_rate)}.")

print(f"My center of gravity is {top_artist}, {top_genre}, and {top_source}.")
print(f"My most common listening time block is {top_block}, and your most common session type is {top_session}.")

## Listening History Window
Streaming habits can change dramatically across months or years. A short listening window captures my recent preferences, while a longer window captures more stable habits. This section measures the total time span represented by the listening history and sets the behavioral window being analyzed.

In [ ]:
show_table(identity, "Listening Identity", 20)

In [ ]:
if not profile.empty:
    if "score" in profile.columns:
        show_table(profile.sort_values("score", ascending=False), "Profile Metrics", 20)
    else:
        show_table(profile, "Profile Metrics", 20)

In [ ]:
if "played_at" in plays.columns and plays["played_at"].notna().any():
    plays["month_clean"] = plays["played_at"].dt.to_period("M").dt.to_timestamp()
elif "month" in plays.columns:
    plays["month_clean"] = pandas.to_datetime(plays["month"], errors="coerce")
else:
    plays["month_clean"] = pandas.NaT

if "month_clean" in plays.columns and plays["month_clean"].notna().any():
    month = (
        plays.groupby("month_clean")
        .agg(
            plays=("event_id", "count") if "event_id" in plays.columns else ("month_clean", "size"),
            tracks=("track_key", "nunique") if "track_key" in plays.columns else ("month_clean", "size"),
            artists=("artist", "nunique") if "artist" in plays.columns else ("month_clean", "size"),
            minutes=("minutes", "sum") if "minutes" in plays.columns else ("month_clean", "size"),
            full_rate=("full_play", "mean") if "full_play" in plays.columns else ("month_clean", "size"),
            friction_rate=("friction", "mean") if "friction" in plays.columns else ("month_clean", "size"),
        )
        .reset_index()
    )

    month.to_csv(tab_root / "exploratory_monthly_listening.csv", index=False)

    chart = plot.line(
        month,
        x="month_clean",
        y="plays",
        markers=True,
        hover_data=["tracks", "artists", "minutes", "full_rate", "friction_rate"],
    )

    show_chart(
        chart,
        "monthly_listening_volume",
        "Listening Volume Over Time",
        "Monthly plays with track, artist, minute, full listen, and friction context.",
    )

    busiest = month.sort_values("plays", ascending=False).head(1)

    if not busiest.empty:
        print(f"Your busiest month was {busiest['month_clean'].iloc[0]:%B %Y}.")

if not days.empty and has_cols(days, ["date", "plays", "minutes"]):
    day_view = days.sort_values("date").copy()

    chart = plot.area(
        day_view,
        x="date",
        y="minutes",
        hover_data=["plays", "tracks", "artists", "top_artist", "top_genre", "full_rate", "friction_rate"],
    )

    show_chart(
        chart,
        "daily_minutes",
        "Daily Listening Minutes",
        "A soft timeline of how much attention music received each day.",
    )

    strongest = day_view.sort_values("minutes", ascending=False).head(1)

    if not strongest.empty:
        print(f"Your strongest listening day was {strongest['date'].iloc[0]:%Y-%m-%d}.")


In [ ]:
if has_cols(hours, ["weekday", "weekday_num", "hour", "plays"]):
    hour_view = hours.copy()
    order = hour_view.sort_values("weekday_num")[["weekday_num", "weekday"]].drop_duplicates()
    heat = hour_view.pivot_table(index="weekday", columns="hour", values="plays", aggfunc="sum").reindex(order["weekday"])

    chart = graph.Figure(
        data=graph.Heatmap(
            z=heat.values,
            x=heat.columns,
            y=heat.index,
            colorscale="Blues",
            hovertemplate="Day: %{y}<br>Hour: %{x}:00<br>Plays: %{z:,}<extra></extra>",
        )
    )

    show_chart(
        chart,
        "weekly_hour_heatmap",
        "Listening Heatmap",
        "The weekly rhythm of when I actually listen to music.",
        560,
    )

    strongest = hour_view.sort_values("plays", ascending=False).head(1)

    if not strongest.empty:
        print(f"My strongest listening pocket is {strongest['weekday'].iloc[0]} at {int(strongest['hour'].iloc[0])}:00.")

In [ ]:
if "time_block" in plays.columns:
    block = (
        plays.groupby("time_block")
        .agg(
            plays=("event_id", "count") if "event_id" in plays.columns else ("time_block", "size"),
            minutes=("minutes", "sum") if "minutes" in plays.columns else ("time_block", "size"),
            full_rate=("full_play", "mean") if "full_play" in plays.columns else ("time_block", "size"),
            friction_rate=("friction", "mean") if "friction" in plays.columns else ("time_block", "size"),
        )
        .reset_index()
        .sort_values("plays", ascending=False)
    )

    chart = plot.bar(
        block,
        x="time_block",
        y="plays",
        hover_data=["minutes", "full_rate", "friction_rate"],
        color="minutes",
        color_continuous_scale="Blues",
    )

    show_chart(
        chart,
        "time_block_listening",
        "Listening by Time Block",
        "Morning, afternoon, evening, and night listening compared.",
    )

    show_table(block, "Time Block Summary", 10)

In [ ]:
if has_cols(artists, ["artist", "plays", "full_rate", "friction_rate", "minutes"]):
    artist_view = artists.sort_values("plays", ascending=False).head(50).copy()

    chart = plot.scatter(
        artist_view,
        x="friction_rate",
        y="full_rate",
        size="plays",
        color="minutes",
        color_continuous_scale="Blues",
        hover_name="artist",
        hover_data=["tracks", "albums", "genres", "active_days", "source_top"],
    )

    chart.update_xaxes(tickformat=".0%", title="Friction Rate")
    chart.update_yaxes(tickformat=".0%", title="Full Listen Rate")

    show_chart(
        chart,
        "artist_galaxy",
        "Artist Galaxy",
        "Each artist is positioned by friction and full-listen behavior, with size showing play volume.",
        620,
    )

    show_table(artists.sort_values("plays", ascending=False), "Core Artists", 20)

    print(f"My most played artist, shockingly, is {top_artist}.")

## Genre Map
While genre labels are imperfect and pretty inconsistent but they are useful for understanding broad listening patterns. This section explores genre concentration, diversity, listening volume, and behavioral quality metrics across all genres.

In [ ]:
if has_cols(genres, ["genre", "plays", "tracks", "artists", "minutes", "share"]):
    genre_view = genres.sort_values("plays", ascending=False).head(20).copy()

    chart = plot.bar(
        genre_view,
        x="share",
        y="genre",
        orientation="h",
        hover_data=["plays", "tracks", "artists", "minutes", "full_rate", "friction_rate", "source_top", "time_top"],
        color="full_rate",
        color_continuous_scale="Blues",
    )

    chart.update_xaxes(tickformat=".0%", title="Share of Plays")
    chart.update_yaxes(categoryorder="total ascending", title="Genre")

    show_chart(
        chart,
        "genre_equalizer",
        "Genre Equalizer",
        "Genre share with full-listen quality and behavioral context.",
        650,
    )

    show_table(genre_view, "Genre Summary", 20)

    print(f"My leading genre is {top_genre}.")

## Source Paths

In [ ]:
if has_cols(sources, ["source_label", "plays", "discovery_share", "repeat_share", "full_rate", "friction_rate"]):
    source_view = sources.sort_values("plays", ascending=False).copy()

    chart = plot.scatter(
        source_view,
        x="discovery_share",
        y="repeat_share",
        size="plays",
        color="friction_rate",
        color_continuous_scale="Blues",
        hover_name="source_label",
        hover_data=["source_role", "tracks", "artists", "genres", "minutes", "full_rate"],
    )

    chart.update_xaxes(tickformat=".0%", title="Discovery Share")
    chart.update_yaxes(tickformat=".0%", title="Repeat Share")

    show_chart(
        chart,
        "source_route_map",
        "Source Route Map",
        "Where listening starts, and whether those routes lead to discovery or repeat behavior.",
        600,
    )

    show_table(source_view, "Source Route Summary", 20)

    print(f"The main source route for me is {top_source}.")

## Sessions

In [ ]:
if has_cols(sessions, ["session_type", "plays", "minutes", "full_rate", "friction_rate", "start_hour"]):
    session_types = (
        sessions.groupby("session_type")
        .agg(
            sessions=("session", "count") if "session" in sessions.columns else ("session_type", "size"),
            plays=("plays", "sum"),
            minutes=("minutes", "sum"),
            average_minutes=("minutes", "mean"),
            full_rate=("full_rate", "mean"),
            friction_rate=("friction_rate", "mean"),
        )
        .reset_index()
        .sort_values("plays", ascending=False)
    )

    session_types.to_csv(tab_root / "exploratory_session_types.csv", index=False)

    chart = plot.bar(
        session_types,
        x="session_type",
        y="plays",
        color="friction_rate",
        color_continuous_scale="Blues",
        hover_data=["sessions", "minutes", "average_minutes", "full_rate"],
    )

    show_chart(
        chart,
        "session_types",
        "Session Types",
        "How your listening behaves when events are grouped into sessions.",
    )

    chart = plot.scatter(
        sessions,
        x="minutes",
        y="friction_rate",
        size="plays",
        color="full_rate",
        color_continuous_scale="Blues",
        hover_name="top_artist",
        hover_data=["session_type", "top_genre", "top_source", "start_hour", "tracks", "artists"],
    )

    chart.update_xaxes(title="Session Minutes")
    chart.update_yaxes(tickformat=".0%", title="Friction Rate")

    show_chart(
        chart,
        "session_behavior_field",
        "Session Behavior Field",
        "Longer sessions, friction, full-listen quality, and dominant artists in one view.",
        620,
    )

    show_table(session_types, "Session Type Summary", 20)

    print(f"My most common session pattern is {top_session}.")

## Lifecycle of Tracks

In [ ]:
if has_cols(life, ["stage", "plays", "minutes", "active_days", "full_rate", "friction_rate", "affinity_score"]):
    stage = (
        life.groupby("stage")
        .agg(
            tracks=("track_key", "count") if "track_key" in life.columns else ("stage", "size"),
            plays=("plays", "sum"),
            minutes=("minutes", "sum"),
            active_days=("active_days", "mean"),
            full_rate=("full_rate", "mean"),
            friction_rate=("friction_rate", "mean"),
            affinity_score=("affinity_score", "mean"),
        )
        .reset_index()
        .sort_values("tracks", ascending=False)
    )

    stage.to_csv(tab_root / "exploratory_lifecycle_stage.csv", index=False)

    chart = plot.bar(
        stage,
        x="stage",
        y="tracks",
        color="affinity_score",
        color_continuous_scale="Blues",
        hover_data=["plays", "minutes", "active_days", "full_rate", "friction_rate"],
    )

    show_chart(
        chart,
        "track_lifecycle",
        "Track Lifecycle",
        "How tracks move from discovery into retention, friction, or dormancy.",
    )

    chart = plot.scatter(
        life,
        x="active_days",
        y="affinity_score",
        size="plays",
        color="friction_rate",
        color_continuous_scale="Blues",
        hover_name="display",
        hover_data=["stage", "artist", "genre", "full_rate", "candidate_class"],
    )

    show_chart(
        chart,
        "lifecycle_field",
        "Lifecycle Field",
        "Track age, affinity, play volume, and friction together.",
        640,
    )

    show_table(stage, "Lifecycle Summary", 20)

    largest_stage = stage.sort_values("tracks", ascending=False).head(1)

    if not largest_stage.empty:
        print(f"The largest lifecycle group is {largest_stage['stage'].iloc[0]}.")

### Behavior of Tracks

In [ ]:
if has_cols(features, ["display", "plays", "full_rate", "friction_rate", "affinity_score", "active_days"]):
    track_view = features[features["plays"] >= 3].copy()

    chart = plot.scatter(
        track_view,
        x="plays",
        y="affinity_score",
        size="active_days",
        color="friction_rate",
        color_continuous_scale="Blues",
        hover_name="display",
        hover_data=["artist", "album", "genre", "minutes", "full_rate", "candidate_class", "source_top"],
    )

    show_chart(
        chart,
        "track_affinity_field",
        "Track Affinity Field",
        "Songs positioned by repeat volume, affinity, active days, and friction.",
        640,
    )

    favorites = features.sort_values(["affinity_score", "plays"], ascending=False).head(25)

    favorite_cols = [
        "display",
        "artist",
        "genre",
        "plays",
        "minutes",
        "active_days",
        "full_rate",
        "friction_rate",
        "affinity_score",
        "candidate_class",
        "source_top",
    ]

    favorite_cols = [col for col in favorite_cols if col in favorites.columns]

    show_table(favorites[favorite_cols], "Highest Affinity Tracks", 25)

    if not favorites.empty:
        print(f"My highest affinity track is {favorites['display'].iloc[0]}.")

## Friction Zone

In [ ]:
if has_cols(features, ["display", "plays", "friction_rate", "full_rate", "artist", "genre"]):
    friction = (
        features[features["plays"] >= 5]
        .sort_values(["friction_rate", "plays"], ascending=False)
        .head(25)
        .copy()
    )

    chart = plot.bar(
        friction,
        x="friction_rate",
        y="display",
        orientation="h",
        color="plays",
        color_continuous_scale="Blues",
        hover_data=["artist", "genre", "full_rate", "minutes", "candidate_class"],
    )

    chart.update_xaxes(tickformat=".0%", title="Friction Rate")
    chart.update_yaxes(categoryorder="total ascending", title="Track")

    show_chart(
        chart,
        "static_zone",
        "The Static Zone",
        "Tracks with enough exposure and the strongest friction signals.",
        760,
    )

    friction_cols = [
        "display",
        "artist",
        "genre",
        "plays",
        "minutes",
        "full_rate",
        "friction_rate",
        "candidate_class",
        "ready_reason",
    ]

    friction_cols = [col for col in friction_cols if col in friction.columns]

    show_table(friction[friction_cols], "High Friction Tracks", 25)

    if not friction.empty:
        print(f"The strongest repeated friction signal is {friction['display'].iloc[0]}.")

## Discovery

In [ ]:
if has_cols(features, ["first_play", "retained", "plays", "display"]):
    discovery = features[features["first_play"].notna()].copy()
    discovery["first_month"] = discovery["first_play"].dt.to_period("M").dt.to_timestamp()

    discovery_month = (
        discovery.groupby("first_month")
        .agg(
            new_tracks=("track_key", "count") if "track_key" in discovery.columns else ("display", "count"),
            retained_tracks=("retained", "sum"),
            average_plays=("plays", "mean"),
            average_affinity=("affinity_score", "mean") if "affinity_score" in discovery.columns else ("plays", "mean"),
        )
        .reset_index()
    )

    discovery_month["retention_rate"] = discovery_month["retained_tracks"] / discovery_month["new_tracks"].replace(0, numpy.nan)

    discovery_month.to_csv(tab_root / "exploratory_discovery_month.csv", index=False)

    chart = plot.bar(
        discovery_month,
        x="first_month",
        y="new_tracks",
        color="retention_rate",
        color_continuous_scale="Blues",
        hover_data=["retained_tracks", "average_plays", "average_affinity"],
    )

    chart.update_yaxes(title="New Tracks")

    show_chart(
        chart,
        "discovery_timeline",
        "Discovery Timeline",
        "New tracks by first-listen month, colored by later retention.",
    )

    best_month = discovery_month.sort_values("retention_rate", ascending=False).head(1)

    if not best_month.empty:
        print(f"My strongest discovery retention month was {best_month['first_month'].iloc[0]:%B %Y}.")

In [ ]:
## Recommendations

In [ ]:
if not recommend_profile.empty:
    show_table(recommend_profile, "Recommendation System Profile", 20)

    if has_cols(recommend_profile, ["metric", "value"]) and recommend_profile["metric"].eq("ready_tracks").any():
        ready_tracks = recommend_profile.loc[recommend_profile["metric"].eq("ready_tracks"), "value"].iloc[0]
    else:
        ready_tracks = numpy.nan

    print(f"The recommendation layer has {number_text(ready_tracks)} ready tracks.")

In [ ]:
if has_cols(candidates, ["candidate_class", "recommend_role", "recommend_score", "quality_score", "risk_score"]):
    candidate_class = (
        candidates.groupby(["candidate_class", "recommend_role"])
        .agg(
            tracks=("track_key", "count") if "track_key" in candidates.columns else ("candidate_class", "size"),
            plays=("plays", "sum"),
            recommend_score=("recommend_score", "mean"),
            quality_score=("quality_score", "mean"),
            risk_score=("risk_score", "mean"),
        )
        .reset_index()
        .sort_values("tracks", ascending=False)
    )

    chart = plot.bar(
        candidate_class,
        x="candidate_class",
        y="tracks",
        color="recommend_role",
        hover_data=["plays", "recommend_score", "quality_score", "risk_score"],
    )

    show_chart(
        chart,
        "recommendation_candidate_classes",
        "Recommendation Candidate Classes",
        "How the recommendation layer labels the track library.",
        620,
    )

    show_table(candidate_class, "Recommendation Candidate Summary", 20)

In [ ]:
if has_cols(recommendations, ["seed_display", "candidate_display", "similarity", "recommend_score", "candidate_class"]):
    chart = plot.histogram(
        recommendations,
        x="similarity",
        nbins=40,
        color="candidate_class",
        hover_data=["recommend_role"],
    )

    chart.update_xaxes(title="Cosine Similarity")

    show_chart(
        chart,
        "recommendation_similarity_distribution",
        "Recommendation Similarity Distribution",
        "Similarity scores across saved similar-track pairs.",
    )

    strongest_pairs = recommendations.sort_values(["similarity", "recommend_score"], ascending=False).head(20)

    pair_cols = [
        "seed_display",
        "candidate_display",
        "similarity",
        "recommend_role",
        "recommend_score",
        "affinity_score",
        "plays",
        "full_rate",
        "friction_rate",
        "candidate_class",
        "recommend_reason",
    ]

    pair_cols = [col for col in pair_cols if col in strongest_pairs.columns]

    show_table(strongest_pairs[pair_cols], "Strongest Recommendation Pairs", 20)

    if not strongest_pairs.empty:
        print(f"The strongest saved recommendation pair is {strongest_pairs['seed_display'].iloc[0]} into {strongest_pairs['candidate_display'].iloc[0]}.")

## **Conclusion**
Even though there are thousands of tracks in the dataset, only a small subset become lasting favorites. Most tracks fall into dropped, observed, or unstable lifecycle stages, suggesting that discovery activity is high but retention is selective. The recommendation layer is supported by a large behavioral feature space and identifies over one thousand tracks with enough signal to make meaningful recommendations. Overall, the data suggests a listener who explores broadly, filters aggressively, and returns repeatedly to a relatively stable co

In [ ]:
summary = pandas.DataFrame(
    [
        {
            "question": "Who am I as a listener?",
            "answer": f"I am a {listener}, and my listening behavior is {selectivity}.",
        },
        {
            "question": "What anchors my taste?",
            "answer": f"My center of gravity is {top_artist}, {top_genre}, and {top_source}.",
        },
        {
            "question": "When do I listen most?",
            "answer": f"My most common listening time block is {top_block}.",
        },
        {
            "question": "How do my sessions behave?",
            "answer": f"My most common session type is {top_session}.",
        },
        {
            "question": "How much data supports this?",
            "answer": f"The analysis uses {total:,} listening events, {track_count:,} tracks, {artist_count:,} artists, {genre_count:,} genres, and {active_days:,} active days.",
        },
        {
            "question": "How real is the recommender?",
            "answer": f"The recommendation tables contain {len(candidates):,} candidates and {len(recommendations):,} saved similar-track pairs.",
        },
    ]
)

summary.to_csv(tab_root / "exploratory_personal_summary.csv", index=False)
show_table(summary, "Personal Summary", 20)